# PC biplots

Pairwise scatterplot matrix (SPLOM) of the projected PCs -- reference (1000G) colored by population, AoU overlaid as a background cloud. Works against either round 1 or round 2 output, whichever paths are set below.

**Performance:** AoU can be 400k+ points; plotting all of them is slow to render and produces bloated files for very little visual gain over a random subsample. Two things keep this fast regardless of `N_PCS_TO_PLOT`: AoU points are randomly subsampled to `MAX_AOU_POINTS`, and every scatter layer is drawn `rasterized=True` (matplotlib flattens the point cloud into a bitmap, so all the axes/labels/legend stay crisp vector text/lines but the expensive part -- drawing potentially tens of thousands of markers -- is cheap and produces a small file). The pairwise grid itself is O(`N_PCS_TO_PLOT`^2) panels, so keep that modest too (6 is already 15 scatter panels).

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

## Inputs

In [ ]:
REF_PCS_PATH = "PLACEHOLDER"    # 1kg_all_pca_reprojected.sscore (round 1) or r2_pca_<cluster>_reprojected.sscore (round 2)
REF_PANEL_PATH = "PLACEHOLDER"  # integrated_call_samples_v3.20130502.ALL.panel
AOU_PCS_PATH = "PLACEHOLDER"    # acaf_projected.sscore (round 1) or r2_projected_<cluster>.sscore (round 2)
OUT_DIR = "PLACEHOLDER"         # figure written here

N_PCS_TO_PLOT = 6      # panel grid is N_PCS_TO_PLOT^2 -- grows fast, keep this modest
MAX_AOU_POINTS = 20000  # random subsample cap for plotting only, not for any analysis
RANDOM_SEED = 0

os.makedirs(OUT_DIR, exist_ok=True)

## Load reference PCs + population labels, build the population palette

Same superpopulation-grouped palette as `round1_filter.ipynb`/`round2_filter.ipynb` -- one colormap per continent, distinguishable shades within it.

In [ ]:
pc_cols = [f"PC{i}_AVG" for i in range(1, N_PCS_TO_PLOT + 1)]

ref = pd.read_csv(REF_PCS_PATH, sep=r"\s+")
ref_id_col = "#IID" if "#IID" in ref.columns else "IID"
ref = ref.rename(columns={ref_id_col: "sample"})

labels = pd.read_csv(REF_PANEL_PATH, sep="\t")
ref = ref.merge(labels[["sample", "pop", "super_pop"]], on="sample")
assert all(c in ref.columns for c in pc_cols), f"missing PC columns, have: {list(ref.columns)}"

SUPERPOP_POPS = {
    "EUR": ["CEU", "TSI", "FIN", "GBR", "IBS"],
    "AFR": ["YRI", "LWK", "GWD", "MSL", "ESN", "ASW", "ACB"],
    "EAS": ["CHB", "JPT", "CHS", "CDX", "KHV"],
    "SAS": ["GIH", "PJL", "BEB", "STU", "ITU"],
    "AMR": ["MXL", "PUR", "CLM", "PEL"],
}
SUPERPOP_CMAPS = {"EUR": "Blues", "AFR": "Oranges", "EAS": "Greens", "SAS": "Purples", "AMR": "Reds"}


def build_pop_colors(superpop_pops=SUPERPOP_POPS, superpop_cmaps=SUPERPOP_CMAPS, lo=0.35, hi=0.85):
    colors = {}
    for superpop, pops in superpop_pops.items():
        cmap = matplotlib.colormaps[superpop_cmaps[superpop]]
        shades = np.linspace(lo, hi, len(pops)) if len(pops) > 1 else [(lo + hi) / 2]
        for pop, frac in zip(pops, shades):
            colors[pop] = mcolors.to_hex(cmap(frac))
    return colors


POP_COLORS = build_pop_colors()
print(f"Reference: {len(ref)} samples, {ref['pop'].nunique()} populations")

## Load + subsample AoU

Subsample happens once here, up front -- every panel in the grid reuses the same `aou_sub`, rather than each panel drawing its own random subset (which would make different panels show different, incomparable subsets of samples).

In [ ]:
t0 = time.time()
aou = pd.read_csv(AOU_PCS_PATH, sep=r"\s+")
aou_id_col = "#IID" if "#IID" in aou.columns else "IID"
aou = aou.rename(columns={aou_id_col: "sample"})
assert all(c in aou.columns for c in pc_cols), f"missing PC columns, have: {list(aou.columns)}"
print(f"AoU: {len(aou)} samples loaded in {time.time() - t0:.1f}s")

if len(aou) > MAX_AOU_POINTS:
    aou_sub = aou.sample(n=MAX_AOU_POINTS, random_state=RANDOM_SEED)
else:
    aou_sub = aou
print(f"Plotting {len(aou_sub)} AoU points (subsampled from {len(aou)})")

## Pairwise scatterplot matrix

Lower triangle only (upper triangle would just be the same pairs transposed -- redundant). Diagonal shows each PC's marginal distribution (reference vs AoU) instead of a scatter against itself. AoU drawn first (background), reference drawn on top (foreground, since it's the labeled/interpretable set) -- both layers rasterized regardless of point count.

In [ ]:
t0 = time.time()
n = len(pc_cols)
fig, axes = plt.subplots(n, n, figsize=(2.2 * n, 2.2 * n))

for row in range(n):
    for col in range(n):
        ax = axes[row, col]
        if row < col:
            ax.axis("off")
            continue

        y_col, x_col = pc_cols[row], pc_cols[col]

        if row == col:
            ax.hist(ref[x_col], bins=40, density=True, color="grey", alpha=0.6, label="1000G")
            ax.hist(aou_sub[x_col], bins=40, density=True, color="black", alpha=0.4, label="AoU")
            if row == 0:
                ax.legend(fontsize=6, loc="upper right")
        else:
            ax.scatter(
                aou_sub[x_col], aou_sub[y_col],
                c="lightgrey", s=3, alpha=0.3, linewidths=0, rasterized=True,
            )
            for pop, grp in ref.groupby("pop"):
                ax.scatter(
                    grp[x_col], grp[y_col],
                    c=POP_COLORS.get(pop, "black"), s=6, alpha=0.7, linewidths=0, rasterized=True,
                )

        if row == n - 1:
            ax.set_xlabel(x_col, fontsize=8)
        else:
            ax.set_xticklabels([])
        if col == 0:
            ax.set_ylabel(y_col, fontsize=8)
        else:
            ax.set_yticklabels([])
        ax.tick_params(labelsize=6)

handles = [
    plt.Line2D([0], [0], marker="o", linestyle="", color=POP_COLORS[pop], label=pop)
    for pop in sorted(ref["pop"].unique())
]
handles.append(plt.Line2D([0], [0], marker="o", linestyle="", color="lightgrey", label="AoU"))
fig.legend(handles=handles, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=7, title="Population")

plt.tight_layout()
plot_path = os.path.join(OUT_DIR, "pc_biplots.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path} in {time.time() - t0:.1f}s")